# Batch 3 Reproducible OCR and Keyword-Detection Pipeline

This notebook lets you reproduce the technical OCR stage yourself in Google Colab.

**Academic interpretation:** the output identifies candidate textual signals. It does not confirm deception. Human validation is required.


## Step 1 — Install software

Tesseract performs OCR. `pytesseract` connects Python to Tesseract. OpenCV is used for image preprocessing.


In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y tesseract-ocr
!pip install -q pytesseract opencv-python-headless pandas


## Step 2 — Upload the cropped-region ZIP

Upload `batch3_cropped_regions_for_ocr.zip`. The code searches all nested folders, so it avoids the earlier `FileNotFoundError`.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from pathlib import Path
import zipfile, os

zip_candidates = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
if not zip_candidates:
    raise FileNotFoundError("Please upload a ZIP file containing the cropped PNG images.")

zip_path = zip_candidates[0]
extract_dir = Path("batch3_crops")
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

image_paths = []
for root, _, filenames in os.walk(extract_dir):
    for filename in filenames:
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(Path(root) / filename)

image_paths = sorted(image_paths)
print("Images found:", len(image_paths))
print("First image:", image_paths[0] if image_paths else "None")


## Step 3 — Display one crop


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample_path = image_paths[0]
sample = Image.open(sample_path)
plt.figure(figsize=(14, 6))
plt.imshow(sample)
plt.axis("off")
plt.title(sample_path.name)
plt.show()


## Step 4 — Run raw OCR


In [ ]:
import pytesseract

raw_text = pytesseract.image_to_string(sample, config="--oem 3 --psm 6")
print(raw_text)


## Step 5 — Preprocess with OpenCV and run OCR again


In [ ]:
import cv2
import numpy as np

img = cv2.imread(str(sample_path))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
upscaled = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced = clahe.apply(upscaled)
_, thresholded = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

if np.mean(thresholded) < 127:
    thresholded = 255 - thresholded

plt.figure(figsize=(14, 6))
plt.imshow(thresholded, cmap="gray")
plt.axis("off")
plt.title("Preprocessed image")
plt.show()

processed_text = pytesseract.image_to_string(thresholded, config="--oem 3 --psm 6")
print(processed_text)


## Step 6 — Define keyword patterns

The patterns operationalize selected textual candidate signals for the three thesis categories.


In [ ]:
import re

URGENCY_PATTERNS = {
    "time left": r"\btime\s+left\b",
    "limited time": r"\blimited[-\s]?time\b",
    "sale starts in": r"\bsale\s+starts\s+in\b",
    "sold today": r"\bsold\s+today\b",
    "only X left": r"\bonly\s+\d[\d,]*\s+left\b",
    "low stock": r"\blow\s+stock\b",
    "selling fast": r"\bselling\s+fast\b",
    "ends": r"\bends?\s+(?:in\b|\d|Jan\b|Feb\b|Mar\b|Apr\b|May\b|Jun\b|Jul\b|Aug\b|Sep\b|Oct\b|Nov\b|Dec\b)",
}

MISDIRECTION_PATTERNS = {
    "% off": r"\b\d{1,3}\s*%\s*off\b",
    "sale": r"\bsale\b",
    "discount": r"\bdiscount(?:ed)?\b",
    "save": r"\bsave\b",
    "deal": r"\bdeals?\b",
    "coupon": r"\bcoupon\b",
    "code": r"\bcode\b",
    "new member": r"\bnew\s+member\b",
    "guest price": r"\bguest\b",
    "best price": r"\bbest\s+price\b",
    "list price": r"\blist\s+price\b",
    "was": r"\bwas\b",
    "now": r"\bnow\b",
}

OBSTRUCTION_PATTERNS = {
    "sign up": r"\bsign\s+up\b",
    "subscribe": r"\bsubscribe\b",
    "enter your email": r"\benter\s+your\s+email\b",
    "email address": r"\bemail\s+address\b",
    "sign in to checkout": r"\bsign\s+in\s+to\s+checkout\b",
    "unlock access": r"\bunlock\s+access\b",
    "new subscribers": r"\bnew\s+subscribers\b",
}

def detect_patterns(text, pattern_dictionary):
    text = str(text).lower()
    return [label for label, pattern in pattern_dictionary.items()
            if re.search(pattern, text, flags=re.IGNORECASE)]


## Step 7 — Process every crop

The notebook stores raw OCR, preprocessed OCR and keyword matches. OCR quality still requires manual review.


In [ ]:
import pandas as pd

def clean_text(text):
    text = str(text).replace("\x0c", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

rows = []

for path in image_paths:
    img = cv2.imread(str(path))
    if img is None:
        rows.append({
            "Crop_Filename": path.name,
            "OCR_Raw_Text": "",
            "OCR_Preprocessed_Text": "",
            "OCR_Chosen_Text": "",
            "OCR_Quality": "failed",
            "Error": "Image could not be read",
        })
        continue

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    aspect_ratio = img.shape[1] / max(img.shape[0], 1)
    psm = 11 if aspect_ratio > 2.2 else 6

    raw = clean_text(
        pytesseract.image_to_string(rgb, config=f"--oem 3 --psm {psm}")
    )

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    upscaled = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(upscaled)
    _, thresholded = cv2.threshold(
        enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    if np.mean(thresholded) < 127:
        thresholded = 255 - thresholded

    processed = clean_text(
        pytesseract.image_to_string(thresholded, config=f"--oem 3 --psm {psm}")
    )

    # Simple reproducible selection rule: keep the version containing more words.
    raw_words = len(re.findall(r"\b\w+\b", raw))
    processed_words = len(re.findall(r"\b\w+\b", processed))
    chosen = processed if processed_words > raw_words else raw
    chosen_method = "preprocessed" if chosen == processed else "raw"

    rows.append({
        "Crop_Filename": path.name,
        "OCR_Raw_Text": raw,
        "OCR_Preprocessed_Text": processed,
        "OCR_Chosen_Text": chosen,
        "OCR_Method": chosen_method,
        "OCR_Quality": "",  # Manually enter: good, partial, failed, or no_text_needed.
        "Urgency_Keywords": ", ".join(detect_patterns(chosen, URGENCY_PATTERNS)),
        "Misdirection_Keywords": ", ".join(detect_patterns(chosen, MISDIRECTION_PATTERNS)),
        "Obstruction_Keywords": ", ".join(detect_patterns(chosen, OBSTRUCTION_PATTERNS)),
    })

results = pd.DataFrame(rows)
results.head()


## Step 8 — Save the result

Download the CSV, inspect every OCR output, and manually assign OCR quality. Visual-signal coding and human validation are separate steps.


In [ ]:
output_csv = "batch3_ocr_keyword_results_colab.csv"
results.to_csv(output_csv, index=False)
files.download(output_csv)


## Connection to the thesis

- OCR and preprocessing support **Sub-RQ1** by extracting textual candidate signals.
- Explicit keyword rules make the artefact transparent and auditable.
- Manual OCR-quality review and later human category validation support the **human-centred AI** perspective.
- The notebook is part of the **artefact design, demonstration and evaluation** stages of DSRM.
